In [1]:
from collections import defaultdict
import pickle
import os
from tqdm import tqdm

def build_flickr8k_dataset(txt_path, img_dir, out_path_train, out_path_valid):
    img_to_caps = defaultdict(list)

    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split("|")
            if len(parts) < 3:
                continue

            img_name = parts[0]
            caption = parts[2].strip()

            img_to_caps[img_name].append(caption)

    dataset = []

    for img_name, caps in tqdm(img_to_caps.items()):
        if len(caps) < 5:
            continue

        dataset.append({
            "image": os.path.join(img_dir, img_name),
            "sentences": [{"raw": c} for c in caps[:5]]
        })

    import random
    
    random.seed(42)
    # random.shuffle(dataset)
    
    split = int(0.85 * len(dataset))
    
    train_ds = dataset[:split]
    valid_ds = dataset[split:]
    with open(out_path_train, "wb") as f:
            pickle.dump(train_ds, f)
    with open(out_path_valid, "wb") as f:
            pickle.dump(valid_ds, f)
    print(f"Done: {len(train_ds)} - {len(valid_ds)}")


In [2]:
build_flickr8k_dataset(
    "/kaggle/input/datasets/nunenuh/flickr8k/captions.txt",
    "/kaggle/input/datasets/nunenuh/flickr8k/images",
    "train_ds.pkl",
    "valid_ds.pkl"
)


100%|██████████| 8092/8092 [00:00<00:00, 361925.72it/s]

Done: 6877 - 1214


In [3]:
import requests
import torch
from PIL import Image
# from transformers import *
from tqdm import tqdm
import pickle
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
import urllib.parse as parse
import os
# from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import nltk
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

In [4]:
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("The device used is", device)

The device used is cuda


In [5]:

# ──────────────────────────────────────────────
# Utility functions
# ──────────────────────────────────────────────

def is_url(string):
    try:
        result = parse.urlparse(string)
        return all([result.scheme, result.netloc, result.path])
    except:
        return False
def load_image(image_path):
    if is_url(image_path):
        return Image.open(path).convert("RGB")   # 🔥 FIX
    elif os.path.exists(image_path):
        return Image.open(image_path).convert("RGB")

In [6]:
import re
import string
from collections import defaultdict

class SimplePTBTokenizer:
    """
    Lightweight PTB-style tokenizer replacement
    (no Java, no Stanford CoreNLP)
    """

    def __init__(self):
        # giống COCO punctuation set
        self.punctuations = set([
            "''", "'", "``", "`",
            "-LRB-", "-RRB-", "-LCB-", "-RCB-",
            ".", "?", "!", ",", ":", "-", "--", "...", ";"
        ])

    def _normalize(self, text):
        # lower case
        text = text.lower()

        # replace multiple spaces
        text = re.sub(r'\s+', ' ', text)

        # remove punctuation (simple version)
        text = text.translate(str.maketrans('', '', string.punctuation))

        return text.strip()

    def tokenize(self, captions):
        """
        captions_for_image:
            dict {
                image_id: [{"caption": str}, ...]
            }
        """

        final = defaultdict(list)

     
        # normalize
        text = self._normalize(captions)

        # tokenize by space
        tokens = text.split()

        # remove PTB-style special tokens if exist
        tokens = [t for t in tokens if t not in self.punctuations]

        # final[img_id].append(" ".join(tokens))

        return tokens


In [7]:
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


In [8]:
# ──────────────────────────────────────────────
# Metric computation
#
# COCO chuẩn: mỗi ảnh có 5 reference captions.
# Với beam search ta sinh ra 5 predicted captions.
# Chiến lược đánh giá:
#   - top1 : chỉ dùng caption đầu tiên (beam tốt nhất)
#   - best  : chọn caption có BLEU-4 cao nhất trong 5 caption (oracle)
# Cả hai đều so với đủ 5 reference captions.
# ──────────────────────────────────────────────

# def _tokenize(s):
#     return s.lower().split()
def to_ptb_format(captions_list):
    return {
        0: [{"caption": c} for c in captions_list]
    }
def compute_metrics(all_preds, all_refs):
    """
    all_preds: list of list[str]
        - mỗi image: 1 hoặc nhiều captions (beam search)
    all_refs: list of list[str]
        - mỗi image: 5 reference captions
    """
    token = SimplePTBTokenizer()
    smooth = SmoothingFunction().method1
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    preds = tokenizer.batch_decode(all_preds, skip_special_tokens=True)
    refs = tokenizer.batch_decode(all_refs, skip_special_tokens=True)
    print(preds)
    # =========================
    # CHUẨN HÓA INPUT
    # =========================
    pred = preds[0]

    pred_tok = token.tokenize(pred)
    bleu1, bleu2, bleu3, bleu4 = [], [], [], []
    rougeL, meteor = [], []
    # ✔ multi-reference format
    refs_tok = [token.tokenize(r) for r in refs]

    bleu1.append(sentence_bleu(refs_tok, pred_tok,
                               weights=(1,0,0,0),
                               smoothing_function=smooth))

    bleu2.append(sentence_bleu(refs_tok, pred_tok,
                               weights=(0.5,0.5,0,0),
                               smoothing_function=smooth))

    bleu3.append(sentence_bleu(refs_tok, pred_tok,
                               weights=(1/3,1/3,1/3,0),
                               smoothing_function=smooth))

    bleu4.append(sentence_bleu(refs_tok, pred_tok,
                               weights=(0.25,0.25,0.25,0.25),
                               smoothing_function=smooth))
    rougeL.append(sum(rouge.score(r, pred)["rougeL"].fmeasure for r in refs_tok[0]) / len(refs_tok[0]))
    meteor.append(meteor_score(refs_tok,pred_tok))
    return {
        "bleu1": round(np.mean(bleu1), 4),
        "bleu2": round(np.mean(bleu2), 4),
        "bleu3": round(np.mean(bleu3), 4),
        "bleu4": round(np.mean(bleu4), 4),
        "rougeL": round(np.mean(rougeL)),
        "meteor": round(np.mean(meteor)),
    }

In [9]:
from transformers import (
    VisionEncoderDecoderModel,
    AutoConfig,
    GPT2TokenizerFast,
    ViTImageProcessor
)

In [10]:
best_model = VisionEncoderDecoderModel.from_pretrained("/kaggle/input/models/thienla/imgcap/pytorch/default/1/img_cap").to(device)
encoder_model = "google/vit-base-patch16-224"
decoder_model = "gpt2"
tokenizer       = GPT2TokenizerFast.from_pretrained(decoder_model)
image_processor = ViTImageProcessor.from_pretrained(encoder_model)
max_length = 32

# GPT-2 specific token config
tokenizer.pad_token                  = tokenizer.eos_token
best_model.config.eos_token_id            = tokenizer.eos_token_id
best_model.config.pad_token_id            = tokenizer.pad_token_id
best_model.config.decoder_start_token_id  = tokenizer.bos_token_id


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/444 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [11]:
# ──────────────────────────────────────────────
# Dataset loading
# ──────────────────────────────────────────────

with open("/kaggle/working/train_ds.pkl", "rb") as f:
    train_ds = pickle.load(f)

with open("/kaggle/working/valid_ds.pkl", "rb") as f:
    valid_ds = pickle.load(f)


In [12]:
# ──────────────────────────────────────────────
# Training configuration
# ──────────────────────────────────────────────
current_step = 0
num_epochs = 6
batch_size = 4
save_steps = 500
num_beams  = 5      # beam search width = số captions sinh ra


In [13]:
from datasets import Dataset

train_ds = Dataset.from_list(train_ds)
valid_ds = Dataset.from_list(valid_ds)
def preprocess(items):
  # preprocess the image
  images   = []
  captions = []

  imgs      = items["image"]     if isinstance(items["image"],     list) else [items["image"]]
  sentences = items["sentences"] if isinstance(items["sentences"], list) else [items["sentences"]]

  for img, sents in zip(imgs, sentences):
        for sentence in sents:
            images.append(load_image(img))
            captions.append(sentence["raw"])
  pixel_values = image_processor(images, return_tensors="pt").pixel_values.to(device)
  # tokenize the caption with truncation and padding
  targets = tokenizer(captions,
                      max_length=max_length, padding="max_length", truncation=True, return_tensors="pt").to(device)
  return {'pixel_values': pixel_values, 'labels': targets["input_ids"]}

train_dataset2 = train_ds.with_transform(preprocess)
valid_dataset2 = valid_ds.with_transform(preprocess)


In [14]:

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch])
    }

In [15]:
train_dataset_loader2 = DataLoader(train_dataset2, collate_fn=collate_fn, batch_size=batch_size, shuffle=True)
valid_dataset_loader2 = DataLoader(valid_dataset2, collate_fn=collate_fn, batch_size=1, shuffle=True)

In [16]:
n_valid_steps = len(valid_dataset_loader2)
n_valid_steps

1214

In [17]:
# from torch.utils.tensorboard import SummaryWriter

# summary_writer = SummaryWriter(log_dir="/kaggle/working/tensorboard")
n_train_steps = num_epochs * len(train_dataset_loader2)
current_step = 0
# logging, eval & save steps
decoder_config = best_model.config.decoder

In [18]:
class IntermediateHead(nn.Module):
    def __init__(self, input_size, output_size):
        super(IntermediateHead, self).__init__()
        self.fc = nn.Linear(input_size, output_size)
        
    def forward(self, x):
        return self.fc(x)
  
vocab_size = 50257
# Create intermediate head modules
num_intermediate_layers = 5
layers_for_exit = [1, 3, 6, 9, 11]
intermediate_heads = nn.ModuleList([IntermediateHead(decoder_config.hidden_size, vocab_size) for _ in range(len(layers_for_exit))])

optimizer = AdamW(intermediate_heads.parameters(), lr=1e-4)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=500, num_training_steps=n_train_steps)

kl_div_loss = torch.nn.KLDivLoss(reduction='batchmean')

In [19]:
# import time
# start = time.time()
# current_step = 0
# T = 2.0  # hoặc 3.0, 5.0
# for epoch in range(num_epochs):
#     # set the model to training model
#     intermediate_heads.train()
#     # initialize the training loss
#     train_loss = 0
#     if current_step >10000:
#           break
#     for batch in tqdm(train_dataset_loader2, "Training", total=len(train_dataset_loader2), leave=False):
#       if current_step >10000:
#           break
#       train_loss = 0    
#       if current_step % save_steps == 0:
#         # evaluate on the validation set
#         print()
#         print(f"Validation at step {current_step}...")
#         print()
#         # set the model to evaluation mode
#         intermediate_heads.eval()
#         # initialize our lists that store the predictions and the labels
#         predictions, labels = [], []
#         # initialize the validation loss
#         valid_loss = 0
#         with torch.no_grad():
#             for batch in valid_dataset_loader2:
#                 # get the batch
#                 pixel_values = batch["pixel_values"]
#                 label_ids = batch["labels"]
#                 # forward pass
#                 outputs = best_model(pixel_values=pixel_values, labels=label_ids, output_hidden_states = True)
#                 int_loss = 0
#                 for exit in range(len(layers_for_exit)):
#                     intermediate_head = intermediate_heads[exit]
#                     intermediate_head = intermediate_head.to(device)
#                     intermediate_logits = intermediate_head(outputs.decoder_hidden_states[layers_for_exit[exit]])
#                     intermediate_loss = F.cross_entropy(intermediate_logits.view(-1, intermediate_logits.size(-1)), label_ids.view(-1))
#                     int_loss+=(layers_for_exit[exit]+1)*intermediate_loss
#                 # get the loss
#                 loss = (int_loss)
#                 # print("int_loss ",int_loss)
#                 valid_loss += loss.item()
            

#         # print the stats
#         print()
#         print(f"Epoch: {epoch}, Step: {current_step}, Train Loss: {train_loss:.4f}, " + 
#               f"Valid Loss: {valid_loss:.4f}")
#         print()
#         intermediate_head.train()
#         torch.cuda.empty_cache()
#         train_loss, valid_loss = 0, 0
#       #training
#       pixel_values = batch["pixel_values"]
#       labels = batch["labels"]
#       # forward pass
#       outputs = best_model(pixel_values=pixel_values, labels=labels, output_hidden_states = True)
#       int_loss_train = 0
#       # w = [i / num_intermediate_layers for i in range(num_intermediate_layers)]
#       for exit in range(len(layers_for_exit)):
#           intermediate_head = intermediate_heads[exit]
#           intermediate_head = intermediate_head.to(device)
#           intermediate_logits = intermediate_head(outputs.decoder_hidden_states[layers_for_exit[exit]])
#           # print('intermediate_logits ',intermediate_logits)
#           # print('outputs.logits ',outputs.logits)
#           # print("teacher logits std:", outputs.logits.std().item())
#           # print("student logits std:", intermediate_logits.std().item())
#           # print("teacher entropy:", -(F.softmax(outputs.logits, -1) * 
#           #                    F.log_softmax(outputs.logits, -1)).sum(-1).mean())

#           compare_head = intermediate_heads[-1]
#           compare_head = compare_head.to(device)
#           compare_head_logits = compare_head(outputs.decoder_hidden_states[-1])
#           # kl_loss = kl_div_loss(F.log_softmax(intermediate_logits, dim=-1), F.softmax(outputs.logits, dim=-1))
#           with torch.cuda.amp.autocast():
#               student_probs = F.log_softmax(intermediate_logits / T, dim=-1)
#               with torch.no_grad():
#                   teacher_probs = F.softmax(outputs.logits / T, dim=-1)  
#               kl_loss = kl_div_loss(
#                 student_probs,
#                 teacher_probs
#                 ) * (T * T)

#           intermediate_loss = (F.cross_entropy(intermediate_logits.view(-1, intermediate_logits.size(-1)), labels.view(-1)))# + 0.8*cosine_loss
#           int_loss_train+=0.5*intermediate_loss + 0.5*kl_loss #+ (current_step / n_train_steps)*cosine_loss_11
#           # Backpropagate the intermediate loss and accumulate gradients
#       # get the loss
#       int_loss_train = int_loss_train / len(layers_for_exit)
#       # backward pass
#       int_loss_train.backward()
#       # update the weights
#       optimizer.step()
#       scheduler.step()
#       # zero the gradients
#       optimizer.zero_grad()
#       loss_v = int_loss_train.item()
#       train_loss += loss_v
#       # increment the step
#       current_step += 1
#       if current_step%(10*save_steps)==0:
#         print(f"Epoch: {epoch}, Step: {current_step}, Train Loss: {train_loss / save_steps:.4f} " )  
#         intermediate_head_weights_dir = f"/kaggle/working/intermediate_head_weights/step-{current_step}"
#         os.makedirs(intermediate_head_weights_dir, exist_ok=True)

#         # Save the weights of each intermediate head
#         for layer_idx, intermediate_head in enumerate(intermediate_heads):
#             head_path = os.path.join(intermediate_head_weights_dir, f"head_layer_{layers_for_exit[layer_idx]}.pt")
#             torch.save(intermediate_head.state_dict(), head_path)
# print('Time train KD and EE: ',time.time()-start)

In [20]:
saved_weights_dir = f"/kaggle/input/models/thienla/kd-ee/pytorch/default/1/img_cap_KD_EE"
best_model = VisionEncoderDecoderModel.from_pretrained(f"/kaggle/input/models/thienla/imgcap/pytorch/default/1/img_cap").to(device)

Loading weights:   0%|          | 0/444 [00:00<?, ?it/s]

In [21]:
valid_dataset_loader = DataLoader(valid_dataset2, collate_fn=collate_fn, batch_size=1, shuffle=True)
def get_evaluation_metrics(model, valid_dataset_loader):
  model.eval()
  n_test_steps = len(valid_dataset_loader)
  # initialize our lists that store the predictions and the labels
  predictions, labels = [], []
  # initialize the test loss
  test_loss = 0.0
  s = 0
  layer_list = []
  bleu1, bleu2, bleu3, bleu4 = 0, 0 ,0,0  
  for idx,batch in enumerate(tqdm(valid_dataset_loader, "Evaluating")):
      if idx == 100:
           break
      # get the batch
      pixel_values = batch["pixel_values"]
      label_ids = batch["labels"]
      # forward pass
      s+=1
      outputs = model(pixel_values=pixel_values, labels=label_ids, output_hidden_states = True)
      int_output = {}
      for layer_idx, intermediate_head in enumerate(intermediate_heads):
        head_path = os.path.join(saved_weights_dir, f"head_layer_{layers_for_exit[layer_idx]}.pt")
        state_dict = torch.load(head_path, map_location=device)
        intermediate_head.load_state_dict(state_dict)
        intermediate_head.to(device)
        output = intermediate_head(outputs.decoder_hidden_states[layers_for_exit[layer_idx]])
        logits = output.detach().cpu()
        int_output[layer_idx] = logits
      # free the GPU memory
      preds = []
      threshold = 1.5
      for i in range(max_length):
        layer_idx = 0
        agg_conf = 0
        for _ in range(len(int_output)):
          prev_logits = None
          logits = int_output[layer_idx]
          logit = logits[0][i].argmax(dim = -1).item()
          logit_2 = outputs.logits.detach().cpu()
          probabilities = torch.softmax(logits, dim=-1)
          confidence = torch.max(probabilities, dim=-1).values
          if prev_logits!= None:
              if prev_logits == logit:
                  agg_conf+=confidence
              else:
                  agg_conf==confidence
          else:
              agg_conf==confidence
          if agg_conf >= threshold:
            preds.append(logit)
            if logit != 50256:
              layer_list.append(layer_idx)
            break
          layer_idx+=1
            
          if layer_idx == len(layers_for_exit):
            preds.append(logit_2[0][i].argmax(dim = -1).item())
            if logit != 50256:
              layer_list.append(layer_idx)
            break
        if logit == 50256:
          layer_list.append(layer_idx)
          break 
      # prediction = []
      # prediction.append(preds)
      # predictions.extend(prediction)
      # labels.extend(label_ids.tolist())
      metric = compute_metrics(preds,label_ids.tolist())
      bleu1+= metric['bleu1']
      bleu2+= metric['bleu2']
      bleu3+= metric['bleu3']
      bleu4+= metric['bleu4']
  bleu1 = bleu1/100 
  bleu2 = bleu2/100 
  bleu3 = bleu3/100 
  bleu4 = bleu4/100
  # eval_prediction = EvalPrediction(predictions=predictions, label_ids=labels)
  # compute the metrics
  # metrics = compute_metrics(predictions,labels)
  return bleu1, bleu2, bleu3, bleu4, layer_list


batch_size = 1
bleu1, bleu2, bleu3, bleu4, layer_list = get_evaluation_metrics(best_model, valid_dataset_loader2)
print()
print("bleu1: ",bleu1)
print("bleu2: ",bleu2)
print("bleu3: ",bleu3)
print("bleu4: ",bleu4)

Evaluating:   0%|          | 0/1214 [00:00<?, ?it/s]

['A dog and tan and and white dog is off a rock of leaves . his in the background .']


Evaluating:   0%|          | 2/1214 [00:18<2:46:24,  8.24s/it]

['A young swim swimming underwater while a toy in his mouth .']


Evaluating:   0%|          | 3/1214 [00:23<2:14:02,  6.64s/it]

...

['A brown dog is a toy toy in in the snow .']


Evaluating:   8%|▊         | 99/1214 [05:03<49:43,  2.68s/it]

['A young boy boy isashes in a p puddle .']


Evaluating:   8%|▊         | 100/1214 [05:07<57:08,  3.08s/it]

['A girl in a pink shirtband plays pink shirt is lookingming . a plastic walls pink pipe . hersticks .']

bleu1:  0.628988
bleu2:  0.406414
bleu3:  0.23035
bleu4:  0.14648999999999995


In [22]:
# import matplotlib.pyplot as plt
# from torchvision import transforms
# model.eval()

# image = load_image('/kaggle/input/datasets/nunenuh/flickr8k/images/1003163366_44323f5815.jpg')
# pixel = image_processor(image, return_tensors="pt").pixel_values.to(device)
# plt.imshow(image)
# plt.show()
# with torch.no_grad():
#     predict = model.generate(pixel,max_length = 32)
# cap_predict = tokenizer.batch_decode(predict[0], skip_special_tokens=True)
# print(cap_predict)